# Create Stiftung Mercator Fellows (FELLOWSHIP PATTERN, method-5 static HTML)

Stiftung Mercator fellow profiles from the foundation's own WordPress-published archive at stiftung-mercator.de. Major German philanthropic foundation funding fellowships in international affairs, climate, education, science, and other policy areas.

**Prerequisites:**
- Run `scripts/local/mercator_fellows_to_s3.py` first.

**Data source:** YOAST sitemap at stiftung-mercator.de/fellow-sitemap*.xml. Two sub-sitemaps total ~1,950 URLs, but each fellow has both EN and DE versions. We scrape only the EN versions for canonical metadata → **~911 unique fellows**.

Each EN fellow page renders labeled fields:
- `<h4 class="single-fellow__info-title">Fellowship</h4>` → program name
- `<h4 class="single-fellow__info-title">Period of the Fellowship</h4>` → "October 2020 - September 2021"
- `<h4 class="single-fellow__info-title">Project title</h4>` → project description
- `<h3 class="single-fellow__biography-title">Short biography</h3>` → bio paragraph

The period field is parsed into `period_start` / `period_end` ISO dates + `start_year` / `end_year` integers.

**S3 location:** `s3a://openalex-ingest/awards/mercator/mercator_fellows.parquet`

**Awarding body:**
- funder_id: 4320327917
- display_name: "Stiftung Mercator"
- country: DE
- ROR: none
- DOI: 10.13039/501100013326

NOT to be confused with Mercator Research Center Ruhr (F4320327916) or Stiftung Mercator Schweiz (F4320321101).

**Coverage (smoke test, 5 fellows, 2026-05-23):**
- 100% fellow_name / slug
- 80% fellowship_program / period_text / start_year / project_title / biography

Full corpus expected to maintain ~80% on the labeled fields; the older fellow profiles (pre-2017) sometimes lack one or more labels.

**Amount/currency:** NULL with §6.7 waiver. Stiftung Mercator publishes overall annual budget (~EUR 80M/year) but does NOT publish per-fellow amount on the API or page. Fellowship-pattern precedent: HHMI #44, CIFAR #79, Damon Runyon #73, Packard #95, Rita Allen #107, Schmidt Sciences #108, NOMIS #109, Wenner-Gren #110.

**Provenance:** `mercator_fellows`.

**Priority:** 116 (UCOP 106 - Velux 115 are immediately-prior slots in flight; Vilcek at 105 is the current main registry head).


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mercator_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/mercator/mercator_fellows.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.mercator_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.mercator_raw;


## Step 1.5: Inspect Raw Data, Money Scan, Native Key

Mercator is non-monetary in our schema (§6.7 waiver). The runbook §1.5 RLIKE money scan is run for completeness; we expect zero matches.

In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.mercator_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.mercator_raw)
WHERE LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT fellow_name, slug, fellowship_program, period_text, start_year,
       SUBSTR(project_title, 1, 100) AS proj_preview,
       SUBSTR(biography, 1, 100) AS bio_preview
FROM openalex.awards.mercator_raw LIMIT 5;


In [ ]:
%sql
-- Native-key uniqueness
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.mercator_raw;


In [ ]:
%sql
-- Year distribution
SELECT start_year, COUNT(*) as fellows
FROM openalex.awards.mercator_raw
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC;


In [ ]:
%sql
-- Top fellowship programs
SELECT fellowship_program, COUNT(*) as fellows
FROM openalex.awards.mercator_raw
GROUP BY fellowship_program
ORDER BY fellows DESC LIMIT 15;


## Step 1.6: Fail-fast — Verify the Mercator Funder Row Exists

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320327917;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mercator_awards
USING delta
AS
WITH
mercator_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320327917
),
raw_prepared AS (
    SELECT
        *,
        TRY_CAST(start_year AS INT) AS parsed_start_year,
        TRY_CAST(end_year AS INT) AS parsed_end_year,
        TRY_TO_DATE(period_start, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(period_end, 'yyyy-MM-dd') AS parsed_end_date
    FROM openalex.awards.mercator_raw
    WHERE fellow_name IS NOT NULL
      AND TRIM(fellow_name) != ''
      AND funder_award_id IS NOT NULL
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        g.fellow_name as display_name,
        g.description as description,

        f.funder_id,
        g.funder_award_id,

        CAST(NULL AS DOUBLE) as amount,
        CAST(NULL AS STRING) as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'fellowship' as funding_type,
        COALESCE(NULLIF(TRIM(g.fellowship_program), ''), 'Stiftung Mercator Fellowship') as funder_scheme,

        'mercator_fellows' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        g.parsed_start_year as start_year,
        g.parsed_end_year as end_year,

        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            CAST(NULL AS STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.landing_page_url as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN mercator_funder f
)
SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 116

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mercator_fellows' AND priority = 116;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    116 as priority
FROM openalex.awards.mercator_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.mercator_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.mercator_awards;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    COUNT(landing_page_url) as has_url,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) as pct_description,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_start_date,
    ROUND(COUNT(lead_investigator) * 100.0 / COUNT(*), 1) as pct_pi
FROM openalex.awards.mercator_awards;


In [ ]:
%sql
SELECT funder_award_id, display_name, funder_scheme, start_year,
       lead_investigator.given_name AS pi_given,
       lead_investigator.family_name AS pi_family,
       landing_page_url
FROM openalex.awards.mercator_awards LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.mercator_awards
GROUP BY funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.mercator_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC;


In [ ]:
%sql
-- §6.7 amount/currency check (waivered — expect 0%)
SELECT COUNT(*) AS total, COUNT(amount) AS has_amount,
       ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
       COUNT(DISTINCT currency) AS distinct_currencies
FROM openalex.awards.mercator_awards;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.mercator_awards
GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 20;


In [ ]:
%sql
SELECT COUNT(*) AS total, COUNT(DISTINCT funder_award_id) AS distinct_ids,
       COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.mercator_awards;
